# Day 3 — 25 Coding Problems (Advanced Python)

### 1. Book class with discount

In [1]:
class Book:
    def __init__(self, title, author, price):
        self.title = title
        self.author = author
        self.price = price

    def apply_discount(self, percent):
        self.price -= self.price * (percent / 100)
        return self.price

Book("Dune", "Herbert", 500).apply_discount(10)

450.0

### 2. Class-level counter

In [2]:
class CountedBook(Book):
    count = 0

    def __init__(self, title, author, price):
        super().__init__(title, author, price)
        CountedBook.count += 1

CountedBook("Dune", "Herbert", 500)
CountedBook("1984", "Orwell", 400)
CountedBook.count

2

### 3. Rectangle equality by area

In [3]:
class Rectangle:
    def __init__(self, w, h):
        self.w, self.h = w, h

    def area(self):
        return self.w * self.h

    def __eq__(self, other):
        return isinstance(other, Rectangle) and self.area() == other.area()

Rectangle(2, 6) == Rectangle(3, 4)

True

### 4. Bank account with deposit/withdraw

In [4]:
class BankAccount:
    def __init__(self, balance=0):
        self.balance = balance

    def deposit(self, amount):
        self.balance += amount

    def withdraw(self, amount):
        if amount > self.balance:
            raise ValueError("Insufficient funds")
        self.balance -= amount

acc = BankAccount(100)
acc.deposit(50)
acc.balance

150

### 5. Custom `__repr__`

In [5]:
class BankAccountRepr(BankAccount):
    def __repr__(self):
        return f"BankAccount(balance={self.balance})"

repr(BankAccountRepr(100))

'BankAccount(balance=100)'

### 6. Shape base class + subclasses

In [6]:
class Shape:
    def area(self):
        raise NotImplementedError

class Circle(Shape):
    def __init__(self, r):
        self.r = r

    def area(self):
        return 3.14159 * self.r ** 2

class Square(Shape):
    def __init__(self, side):
        self.side = side

    def area(self):
        return self.side ** 2

Circle(2).area(), Square(3).area()

(12.56636, 9)

### 7. Total area across shapes (polymorphism)

In [7]:
def total_area(shapes):
    return sum(s.area() for s in shapes)

total_area([Circle(2), Square(3)])

21.56636

### 8. Method overriding with `super()`

In [8]:
class Animal:
    def speak(self):
        return "Some generic sound"

class Cat(Animal):
    def speak(self):
        return super().speak() + " -> Meow!"

Cat().speak()

'Some generic sound -> Meow!'

### 9. Multiple inheritance

In [9]:
class Flyer:
    def fly(self):
        return "Flying"

class Swimmer:
    def swim(self):
        return "Swimming"

class Duck(Flyer, Swimmer):
    pass

Duck().fly(), Duck().swim()

('Flying', 'Swimming')

### 10. `isinstance` / `issubclass`

In [10]:
c = Circle(2)
isinstance(c, Shape), issubclass(Circle, Shape)

(True, True)

### 11. Private attribute with read-only property

In [11]:
class SecureAccount:
    def __init__(self, balance):
        self.__balance = balance

    @property
    def balance(self):
        return self.__balance

SecureAccount(500).balance

500

### 12. Abstract base class with two implementations

In [12]:
from abc import ABC, abstractmethod

class PaymentMethod(ABC):
    @abstractmethod
    def pay(self, amount):
        ...

class CreditCard(PaymentMethod):
    def pay(self, amount):
        return f"Paid {amount} via Credit Card"

class UPI(PaymentMethod):
    def pay(self, amount):
        return f"Paid {amount} via UPI"

CreditCard().pay(100), UPI().pay(50)

('Paid 100 via Credit Card', 'Paid 50 via UPI')

### 13. Abstract class cannot be instantiated

In [13]:
try:
    PaymentMethod()
    result = False
except TypeError:
    result = True

result

True

### 14. Custom iterator — Countdown

In [14]:
class Countdown:
    def __init__(self, start):
        self.current = start

    def __iter__(self):
        return self

    def __next__(self):
        if self.current <= 0:
            raise StopIteration
        val = self.current
        self.current -= 1
        return val

list(Countdown(5))

[5, 4, 3, 2, 1]

### 15. Generator — even numbers

In [15]:
def even_numbers(limit):
    for i in range(0, limit + 1, 2):
        yield i

list(even_numbers(10))

[0, 2, 4, 6, 8, 10]

### 16. Generator — Fibonacci

In [16]:
def fibonacci(n):
    a, b = 0, 1
    for _ in range(n):
        yield a
        a, b = b, a + b

list(fibonacci(8))

[0, 1, 1, 2, 3, 5, 8, 13]

### 17. Generator expression — sum of squares

In [17]:
def sum_of_squares(n=100):
    return sum(i * i for i in range(1, n + 1))

sum_of_squares()

338350

### 18. Generator — chunked list

In [18]:
def chunked(iterable, size):
    for i in range(0, len(iterable), size):
        yield iterable[i:i + size]

list(chunked([1, 2, 3, 4, 5, 6, 7], 3))

[[1, 2, 3], [4, 5, 6], [7]]

### 19. Timer decorator

In [19]:
import functools, time

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        print(f"{func.__name__} took {time.perf_counter() - start:.6f}s")
        return result
    return wrapper

@timer
def slow_add(a, b):
    return a + b

slow_add(2, 3)

slow_add took 0.000001s


5

### 20. Decorator factory — retry

In [20]:
def retry(times):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last_exc = None
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_exc = e
            raise last_exc
        return wrapper
    return decorator

attempts = {"n": 0}

@retry(3)
def flaky():
    attempts["n"] += 1
    if attempts["n"] < 2:
        raise RuntimeError("temporary failure")
    return "success"

flaky()

'success'

### 21. Validation decorator

In [21]:
def require_positive(func):
    @functools.wraps(func)
    def wrapper(n, *args, **kwargs):
        if n <= 0:
            raise ValueError("Argument must be positive")
        return func(n, *args, **kwargs)
    return wrapper

@require_positive
def square_root_demo(n):
    return n ** 0.5

square_root_demo(16)

4.0

### 22. Class-based context manager — Timer

In [22]:
class Timer:
    def __enter__(self):
        self.start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.elapsed = time.perf_counter() - self.start
        print(f"Block took {self.elapsed:.6f}s")
        return False

with Timer():
    total = sum(range(1000000))

total

Block took 0.014710s


499999500000

### 23. Function-based context manager — suppress and log

In [23]:
import logging
from contextlib import contextmanager

logging.basicConfig(level=logging.INFO)
log = logging.getLogger("demo")

@contextmanager
def suppress_and_log(exc_type):
    try:
        yield
    except exc_type as e:
        log.error(f"Suppressed {exc_type.__name__}: {e}")

with suppress_and_log(ZeroDivisionError):
    1 / 0

print("continues normally")

ERROR:demo:Suppressed ZeroDivisionError: division by zero


continues normally


### 24. JSON save/load helpers

In [24]:
import json

def save_json(data, path):
    with open(path, "w") as f:
        json.dump(data, f, indent=2)

def load_json(path):
    with open(path, "r") as f:
        return json.load(f)

save_json({"hello": "world"}, "sample.json")
load_json("sample.json")

{'hello': 'world'}

### 25. REST API consumption

In [25]:
import requests

def fetch_json(url):
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    return response.json()

try:
    print(fetch_json("https://api.github.com/users/octocat")["login"])
except requests.exceptions.RequestException as e:
    print(e)

403 Client Error: Forbidden for url: https://api.github.com/users/octocat
